In [ ]:
import emap
import json
import os

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$andu" or type_ == "$oru" or type_ == "$xoru":
        return len(ports[0]) * 1.0
    if type_ == "$not":
        return len(ports[0]) * 0.5
    raise ValueError(f"Unknown type: {type_}")

def run(test_file: str, top_module: str):
    netlist = emap.NetlistDB(SCHEMA_PATH)
    with open(f"eval/epfl/{test_file}.json", "r") as f:
        netlist.build_from_json(json.load(f)["modules"][top_module])

    # this netlist is already saturated
    netlist.rebuild()

    mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, OutputFlag=True)

    with open(f"epfl/out/{test_file}_extracted.json", "w") as f:
        json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

os.makedirs("eval/out", exist_ok=True)

In [ ]:
run("saturated_dec_nextmap", "top")